# Kapilarni uspon — predvidi, izračunaj, provjeri

**Poglavlje U02: reologija i međupovršinske pojave**

Ravnotežu težine stupca i vertikalne komponente površinske sile koristimo
kao osnovni scenarij. Zatim inverzno tražimo promjer kapilare i procjenjujemo
utjecaj mjernih nesigurnosti.


## 1. Predvidi

1. Hoće li prepolovljen promjer približno udvostručiti visinu uspona?
2. Što se događa pri kontaktnom kutu $90^\circ$, a što iznad njega?
3. Ako želimo uspon od 30 mm, očekuješ li promjer bliži 0,1 mm ili 10 mm?

Zapiši predviđanje prije pokretanja ćelija.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
G = 9.81

def kapilarni_uspon(sigma_N_m, theta_deg, d_m, rho_kg_m3=998.0):
    theta = np.radians(theta_deg)
    return 4.0 * sigma_N_m * np.cos(theta) / (rho_kg_m3 * G * d_m)

sigma, theta, d, rho = 0.0728, 25.0, 0.80e-3, 998.0
h = kapilarni_uspon(sigma, theta, d, rho)
print(f"Osnovni scenarij: h = {1000*h:.2f} mm")


## 2. Izračunaj — inverzni problem bisekcijom

Umjesto izravnog uvrštavanja odabiremo ciljani uspon $h^*=30$ mm i
numerički tražimo promjer. Bisekcija zadržava interval u kojem funkcija
$h(d)-h^*$ mijenja predznak; širina intervala daje izravnu procjenu pogreške.


In [ ]:
def promjer_bisekcijom(h_cilj, sigma, theta_deg, rho,
                       d_lijevo=0.02e-3, d_desno=10e-3,
                       tol=1e-10, max_iter=80):
    f = lambda d: kapilarni_uspon(sigma, theta_deg, d, rho) - h_cilj
    a, b = d_lijevo, d_desno
    if f(a) * f(b) >= 0:
        raise ValueError("Početni interval ne omeđuje rješenje.")
    povijest = []
    for _ in range(max_iter):
        c = 0.5 * (a + b)
        if f(a) * f(c) <= 0:
            b = c
        else:
            a = c
        povijest.append(b - a)
        if b - a < tol:
            break
    return 0.5 * (a + b), np.asarray(povijest)

h_cilj = 30e-3
d_star, sirina = promjer_bisekcijom(h_cilj, sigma, theta, rho)
print(f"d* = {1000*d_star:.6f} mm nakon {len(sirina)} iteracija")
print(f"konačna granica pogreške < {0.5*1000*sirina[-1]:.3e} mm")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.semilogy(np.arange(1, len(sirina)+1), 0.5*sirina, "o-")
ax.set(xlabel="iteracija", ylabel="granica pogreške promjera (m)",
       title="Konvergencija bisekcije")
ax.grid(ls=":", alpha=0.6)
plt.show()


## 3. Provjeri — osjetljivost i propagacija nesigurnosti

Nesigurnost rezultata računamo deterministički iz centralnih konačnih
razlika. Tako ista procedura vrijedi i kada formula postane složenija.


In [ ]:
x0 = np.array([sigma, theta, d, rho])
ux = np.array([0.0005, 1.0, 0.01e-3, 1.0])
korak = np.array([1e-6, 1e-3, 1e-8, 1e-3])

def model_x(x):
    return kapilarni_uspon(x[0], x[1], x[2], x[3])

grad = np.empty(4)
for i in range(4):
    xp, xm = x0.copy(), x0.copy()
    xp[i] += korak[i]
    xm[i] -= korak[i]
    grad[i] = (model_x(xp) - model_x(xm)) / (2*korak[i])
doprinosi = np.abs(grad * ux)
u_h = np.sqrt(np.sum(doprinosi**2))
print(f"h = {1000*h:.2f} ± {1000*u_h:.2f} mm")
print("Doprinosi u_h [mm] za sigma, theta, d, rho:",
      np.round(1000*doprinosi, 3))

# Referentno rješenje inverznog problema i dva granična/scaling testa.
assert abs(kapilarni_uspon(sigma, theta, d_star, rho) - h_cilj) < 1e-8
assert abs(kapilarni_uspon(sigma, 90.0, d, rho)) < 1e-14
assert np.isclose(kapilarni_uspon(sigma, theta, d/2, rho), 2*h,
                  rtol=1e-12)
print("PASS: inverzno rješenje, kut 90° i skaliranje 1/d su potvrđeni.")


## Granica modela

Jednadžba pretpostavlja kružnu kapilaru, statičku ravnotežu, poznat kontaktni
kut i zanemarivu gravitacijsku promjenu zakrivljenosti meniska. Kod vrlo sitnih
ili onečišćenih cijevi histereza kontaktnog kuta može dominirati rezultatom.
